In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import numpy as np

class ConvBNReLU(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResNetBackbone(nn.Module):
    def __init__(self, backbone_name="resnet50"):
        super().__init__()
        if backbone_name not in models.__dict__:
            raise Exception(f"No model named {backbone_name} exists in torchvision.models.")
        backbone = models.__dict__[backbone_name](pretrained=True)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
        )
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        if backbone_name.startswith("resnet50") or backbone_name.startswith("resnet101") or backbone_name.startswith("resnet152"):
            self.out_channels = (256, 512, 1024, 2048)
        else:
            self.out_channels = (64, 128, 256, 512)

    def forward(self, x):
        x = self.stem(x)
        c1 = x
        x = self.maxpool(x)
        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return c2, c3, c4, c5


class BITChangeDetector(nn.Module):
    def __init__(self, backbone="resnet50", embed_dim=256, num_heads=8, depth=6):
        super().__init__()
        self.backbone = ResNetBackbone(backbone)
        c2_channels, c3_channels, c4_channels, c5_channels = self.backbone.out_channels

        self.tokenizer = nn.Conv2d(c5_channels, embed_dim, kernel_size=1, bias=False)
        self.pos_embed = nn.Parameter(torch.zeros(1, embed_dim, 7, 7))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.c4_proj = nn.Conv2d(c4_channels, embed_dim // 2, kernel_size=1, bias=False)
        self.c3_proj = nn.Conv2d(c3_channels, embed_dim // 4, kernel_size=1, bias=False)
        self.c2_proj = nn.Conv2d(c2_channels, embed_dim // 8, kernel_size=1, bias=False)

        self.decoder4 = ConvBNReLU(embed_dim + embed_dim // 2, embed_dim // 2)
        self.decoder3 = ConvBNReLU(embed_dim // 2 + embed_dim // 4, embed_dim // 4)
        self.decoder2 = ConvBNReLU(embed_dim // 4 + embed_dim // 8, embed_dim // 8)
        self.head = nn.Sequential(
            nn.Conv2d(embed_dim // 8, embed_dim // 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(embed_dim // 8),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim // 8, 1, kernel_size=1),
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.pos_embed, mean=0.0, std=0.02)
        nn.init.kaiming_normal_(self.tokenizer.weight, nonlinearity="relu")
        for module in self.head.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1.0)
                nn.init.constant_(module.bias, 0.0)

    def _get_pos_embed(self, h, w):
        if self.pos_embed.shape[-2:] == (h, w):
            return self.pos_embed
        return F.interpolate(self.pos_embed, size=(h, w), mode="bilinear", align_corners=False)

    def forward(self, img1, img2):
        c2_pre, c3_pre, c4_pre, c5_pre = self.backbone(img1)
        c2_post, c3_post, c4_post, c5_post = self.backbone(img2)

        diff_c2 = torch.abs(c2_pre - c2_post)
        diff_c3 = torch.abs(c3_pre - c3_post)
        diff_c4 = torch.abs(c4_pre - c4_post)
        diff_c5 = torch.abs(c5_pre - c5_post)

        tokens = self.tokenizer(diff_c5)
        pos_embed = self._get_pos_embed(tokens.shape[-2], tokens.shape[-1])
        tokens = tokens + pos_embed
        b, c, h, w = tokens.shape
        tokens = tokens.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        tokens = tokens.transpose(1, 2).reshape(b, c, h, w)

        x = F.interpolate(tokens, size=diff_c4.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, self.c4_proj(diff_c4)], dim=1)
        x = self.decoder4(x)

        x = F.interpolate(x, size=diff_c3.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, self.c3_proj(diff_c3)], dim=1)
        x = self.decoder3(x)

        x = F.interpolate(x, size=diff_c2.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, self.c2_proj(diff_c2)], dim=1)
        x = self.decoder2(x)

        x = F.interpolate(x, size=img1.shape[-2:], mode="bilinear", align_corners=False)
        logits = self.head(x)
        return logits


# ============================================================================
# IMPROVED LOSS FUNCTIONS
# ============================================================================
class DiceLoss(nn.Module):
    """Dice Loss for handling class imbalance in segmentation"""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1 - dice


class CombinedLoss(nn.Module):
    """Combined BCE + Dice Loss with class weighting"""
    def __init__(self, pos_weight=None, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
    
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss


# ============================================================================
# DATASET CLASS
# ============================================================================
class EarthquakeDamageDataset(Dataset):
    def __init__(
        self,
        root_dir,
        transform=None,
        damage_threshold=0.05,  # INCREASED from 0.01
        image_files=None,
        minority_ratio_threshold=0.1,
    ):
        self.root_dir = root_dir
        self.pre_dir = os.path.join(root_dir, 'A')
        self.post_dir = os.path.join(root_dir, 'B')
        self.label_dir = os.path.join(root_dir, 'label')
        self.transform = transform
        self.damage_threshold = damage_threshold
        self.minority_ratio_threshold = minority_ratio_threshold

        if image_files is not None:
            self.image_files = list(image_files)
        else:
            self.image_files = sorted(
                [f for f in os.listdir(self.pre_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            )

        self.sample_ratios = []
        self.binary_labels = []
        self.sample_weights = []
        self.pixel_stats = {'changed': 0, 'unchanged': 0}
        
        for filename in self.image_files:
            label_path = os.path.join(self.label_dir, filename)
            label_array = np.array(Image.open(label_path).convert('L'))
            binary_mask = (label_array > 127).astype(np.uint8)
            ratio = float(binary_mask.mean())
            binary_label = 1.0 if ratio > self.damage_threshold else 0.0
            weight = 1.0 if ratio < self.minority_ratio_threshold else 0.5
            
            # Track pixel statistics
            self.pixel_stats['changed'] += binary_mask.sum()
            self.pixel_stats['unchanged'] += (1 - binary_mask).sum()
            
            self.sample_ratios.append(ratio)
            self.binary_labels.append(binary_label)
            self.sample_weights.append(weight)

    def get_pos_weight(self):
        """Calculate positive class weight for BCEWithLogitsLoss"""
        if self.pixel_stats['changed'] == 0:
            return torch.tensor([1.0])
        ratio = self.pixel_stats['unchanged'] / self.pixel_stats['changed']
        return torch.tensor([ratio])

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        filename = self.image_files[idx]
        pre_img_path = os.path.join(self.pre_dir, filename)
        post_img_path = os.path.join(self.post_dir, filename)
        label_path = os.path.join(self.label_dir, filename)

        pre_img = np.array(Image.open(pre_img_path).convert('RGB'))
        post_img = np.array(Image.open(post_img_path).convert('RGB'))
        label_img = np.array(Image.open(label_path).convert('L'))

        mask = (label_img > 127).astype(np.uint8)
        assert pre_img.shape[:2] == mask.shape[:2], (
            f"Image and label size mismatch for {filename}: {pre_img.shape[:2]} vs {mask.shape[:2]}"
        )

        if self.transform:
            augmented = self.transform(image=pre_img, image_post=post_img, mask=mask)
            pre_tensor = augmented['image']
            post_tensor = augmented['image_post']
            mask_tensor = augmented['mask']
        else:
            pre_tensor = torch.from_numpy(pre_img.astype(np.float32) / 255.0).permute(2, 0, 1)
            post_tensor = torch.from_numpy(post_img.astype(np.float32) / 255.0).permute(2, 0, 1)
            mask_tensor = torch.from_numpy(mask).unsqueeze(0)

        if isinstance(mask_tensor, torch.Tensor):
            if mask_tensor.ndim == 2:
                mask_tensor = mask_tensor.unsqueeze(0)
            mask_tensor = mask_tensor.float()
        else:
            mask_tensor = torch.from_numpy(mask_tensor).unsqueeze(0).float()

        class_label = torch.tensor([self.binary_labels[idx]], dtype=torch.float32)
        return pre_tensor.float(), post_tensor.float(), class_label, mask_tensor, filename


# ============================================================================
# AUGMENTATION HELPERS
# ============================================================================
def build_train_transform(final_size=224):
    return A.Compose(
        [
            A.Resize(final_size, final_size),
            A.RandomRotate90(p=0.5),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.0625,
                scale_limit=0.1,
                rotate_limit=15,
                border_mode=0,
                p=0.5,
            ),
            A.OneOf(
                [
                    A.RandomBrightnessContrast(p=1.0),
                    A.RandomGamma(p=1.0),
                    A.HueSaturationValue(p=1.0),
                ],
                p=0.5,
            ),
            A.GaussNoise(p=0.3),
            A.Blur(blur_limit=3, p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ],
        additional_targets={'image_post': 'image'},
    )


def build_eval_transform(final_size=224):
    return A.Compose(
        [
            A.Resize(final_size, final_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ],
        additional_targets={'image_post': 'image'},
    )


# ============================================================================
# DATA LOADERS
# ============================================================================
def get_dataloaders(root_dir, batch_size=32, damage_threshold=0.05, num_workers=4):
    full_dataset = EarthquakeDamageDataset(
        root_dir=root_dir,
        transform=None,
        damage_threshold=damage_threshold,
    )

    # Get positive weight for loss function
    pos_weight = full_dataset.get_pos_weight()
    print(f"\nClass imbalance ratio: {pos_weight.item():.2f}:1 (unchanged:changed)")
    print(f"Total changed pixels: {full_dataset.pixel_stats['changed']:,}")
    print(f"Total unchanged pixels: {full_dataset.pixel_stats['unchanged']:,}")

    total_size = len(full_dataset)
    train_size = int(0.7 * total_size)
    val_size = int(0.15 * total_size)
    test_size = total_size - train_size - val_size

    print(f"\nDataset split:")
    print(f"Total images: {total_size}")
    print(f"Train: {train_size} ({train_size/total_size*100:.1f}%)")
    print(f"Validation: {val_size} ({val_size/total_size*100:.1f}%)")
    print(f"Test: {test_size} ({test_size/total_size*100:.1f}%)")

    generator = torch.Generator().manual_seed(42)
    train_subset, val_subset, test_subset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size, test_size], generator=generator
    )

    def subset_filenames(subset):
        return [full_dataset.image_files[i] for i in subset.indices]

    train_transform = build_train_transform()
    eval_transform = build_eval_transform()

    train_dataset = EarthquakeDamageDataset(
        root_dir=root_dir,
        transform=train_transform,
        damage_threshold=damage_threshold,
        image_files=subset_filenames(train_subset),
    )
    val_dataset = EarthquakeDamageDataset(
        root_dir=root_dir,
        transform=eval_transform,
        damage_threshold=damage_threshold,
        image_files=subset_filenames(val_subset),
    )
    test_dataset = EarthquakeDamageDataset(
        root_dir=root_dir,
        transform=eval_transform,
        damage_threshold=damage_threshold,
        image_files=subset_filenames(test_subset),
    )

    train_weights = torch.as_tensor(train_dataset.sample_weights, dtype=torch.double)
    train_sampler = WeightedRandomSampler(
        weights=train_weights,
        num_samples=len(train_weights),
        replacement=True,
    )

    pin_memory = True
    persistent_workers = num_workers > 0

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
    )

    return train_loader, val_loader, test_loader, pos_weight


# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0

    for batch_idx, (pre_imgs, post_imgs, _, masks, _) in enumerate(train_loader):
        pre_imgs = pre_imgs.to(device, non_blocking=True)
        post_imgs = post_imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).float()

        optimizer.zero_grad()
        logits = model(pre_imgs, post_imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = (torch.sigmoid(logits) > 0.5).float()
        total_pixels += masks.numel()
        correct_pixels += (preds == masks).sum().item()

        if (batch_idx + 1) % 10 == 0:
            print(f'Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct_pixels / total_pixels if total_pixels > 0 else 0.0
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0

    with torch.no_grad():
        for pre_imgs, post_imgs, _, masks, _ in val_loader:
            pre_imgs = pre_imgs.to(device, non_blocking=True)
            post_imgs = post_imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            logits = model(pre_imgs, post_imgs)
            loss = criterion(logits, masks)

            running_loss += loss.item()

            preds = (torch.sigmoid(logits) > 0.5).float()
            total_pixels += masks.numel()
            correct_pixels += (preds == masks).sum().item()

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100 * correct_pixels / total_pixels if total_pixels > 0 else 0.0
    return epoch_loss, epoch_acc


# ============================================================================
# MAIN TRAINING SCRIPT
# ============================================================================
if __name__ == "__main__":
    print("\n" + "="*60)
    print("BIT CHANGE DETECTOR - IMPROVED TRAINING")
    print("With Balanced Loss and Better Thresholds")
    print("="*60)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\nGPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    
    # IMPROVED HYPERPARAMETERS
    BATCH_SIZE = 32
    LEARNING_RATE = 0.0001
    NUM_EPOCHS = 50
    DAMAGE_THRESHOLD = 0.05  # INCREASED: Now requires 5% changed pixels
    NUM_WORKERS = 4
    
    print(f"\nHyperparameters:")
    print(f"Batch size per GPU: {BATCH_SIZE}")
    print(f"Effective batch size: {BATCH_SIZE * torch.cuda.device_count()}")
    print(f"Learning rate: {LEARNING_RATE}")
    print(f"Epochs: {NUM_EPOCHS}")
    print(f"Damage threshold: {DAMAGE_THRESHOLD} (5% changed pixels)")
    
    # Create dataloaders with pos_weight
    print("\nLoading dataset...")
    train_loader, val_loader, test_loader, pos_weight = get_dataloaders(
        root_dir='/kaggle/input/tue-cd/TUE',
        batch_size=BATCH_SIZE,
        damage_threshold=DAMAGE_THRESHOLD,
        num_workers=NUM_WORKERS
    )
    
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    print(f"Test batches: {len(test_loader)}")
    
    # Initialize model
    print("\nInitializing BIT Change Detection Network...")
    model = BITChangeDetector(backbone='resnet152', embed_dim=256, num_heads=8, depth=6)
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
        model = torch.nn.DataParallel(model)
    
    model = model.to(device)
    
    # IMPROVED LOSS: Combined BCE + Dice with class weighting
    print(f"\nUsing Combined Loss (BCE + Dice) with pos_weight={pos_weight.item():.2f}")
    criterion = CombinedLoss(
        pos_weight=pos_weight.to(device),
        bce_weight=0.5,
        dice_weight=0.5
    )
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )
    
    # Training loop
    print("\n" + "=" * 60)
    print("Starting Training...")
    print("=" * 60 + "\n")
    
    best_val_acc = 0.0
    
    for epoch in range(NUM_EPOCHS):
        print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}')
        print('-' * 60)
        
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        print(f'\nTrain Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            model_to_save = model.module if hasattr(model, 'module') else model
            torch.save({
                'epoch': epoch,
                'model_state_dict': model_to_save.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'damage_threshold': DAMAGE_THRESHOLD,
                'pos_weight': pos_weight.item(),
            }, 'best_bit_balanced_model.pth')
            print(f'✓ Saved best model with val_acc: {val_acc:.2f}%')
        
        print('=' * 60)
    
    print("\n" + "=" * 60)
    print(f"Training Complete! Best Val Accuracy: {best_val_acc:.2f}%")
    print("=" * 60)

In [ ]:
import torch
from sklearn.metrics import *
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create model
model = BITChangeDetector(backbone='resnet152', embed_dim=256, num_heads=8, depth=6)
model.to(device)

# Load checkpoint
ckpt = torch.load('best_bit_balanced_model.pth', map_location=device)
state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt

# Auto-fix module prefix
if list(state_dict.keys())[0].startswith('module.'):
    state_dict = {k[7:]: v for k, v in state_dict.items()}  # remove 'module.'

model.load_state_dict(state_dict)

# Optional: use all GPUs
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

model.eval()
print("Model loaded and ready!")
print(f"Best model loaded (epoch {checkpoint.get('epoch', '?')}, val_acc {checkpoint.get('val_acc', '?'):.2f}%)")

# ------------------------------------------------------------
# 2. Re-create test loader (same as training script)
# ------------------------------------------------------------
test_loader = test_loader  # <-- already exists in your environment from previous cell
# If you restarted kernel, uncomment and run the get_dataloaders part again

# ------------------------------------------------------------
# 3. Run inference on test set
# ------------------------------------------------------------
all_probs = []
all_preds = []
all_targets = []
all_filenames = []

print("Running test inference...")
with torch.no_grad():
    for pre_imgs, post_imgs, class_labels, masks, filenames in tqdm(test_loader, desc="Test"):
        pre_imgs = pre_imgs.to(device)
        post_imgs = post_imgs.to(device)
        masks = masks.to(device).float().squeeze(1)  # (B, H, W)

        logits = model(pre_imgs, post_imgs)          # (B, 1, H, W)
        probs = torch.sigmoid(logits).squeeze(1)     # (B, H, W)
        preds = (probs > 0.5).float()

        # Flatten for metrics
        all_probs.append(probs.cpu().numpy().ravel())
        all_preds.append(preds.cpu().numpy().ravel())
        all_targets.append(masks.cpu().numpy().ravel())
        
        all_filenames.extend(filenames)

# Concatenate everything
y_true = np.concatenate(all_targets)
y_pred = np.concatenate(all_preds)
y_prob = np.concatenate(all_probs)

# ------------------------------------------------------------
# 4. Compute all metrics
# ------------------------------------------------------------
acc = accuracy_score(y_true, y_pred)
try:
    auc = roc_auc_score(y_true, y_prob)
except ValueError:
    auc = np.nan  # if only one class present

mae = mean_absolute_error(y_true, y_prob)

precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, labels=[0,1])
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

print("\n" + "="*60)
print("          TEST SET RESULTS (Pixel-level)")
print("="*60)
print(f"Accuracy           : {acc*100:6.2f}%")
print(f"AUC (ROC)          : {auc:6.4f}" if not np.isnan(auc) else "AUC              : N/A (single class)")
print(f"MAE                : {mae:6.4f}")
print(f"Precision  [0/1]   : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall     [0/1]   : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score   [0/1]   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"F1-macro           : {f1_macro:.4f}")
print(f"F1-weighted        : {f1_weighted:.4f}")
print("="*60)

# ------------------------------------------------------------
# 5. Classification Report
# ------------------------------------------------------------
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Unchanged', 'Damaged'], digits=4))

# ------------------------------------------------------------
# 6. Confusion Matrix (pixel-level)
# ------------------------------------------------------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Unchanged', 'Damaged'],
            yticklabels=['Unchanged', 'Damaged'])
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion Matrix (Pixel-level) - Test Set')
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. Optional: Image-level metrics (if you care about "building damaged or not")
# ------------------------------------------------------------
# Re-use the per-image binary labels from dataset
image_true = []
image_pred = []
image_prob = []

with torch.no_grad():
    for pre_imgs, post_imgs, class_labels, masks, filenames in tqdm(test_loader, desc="Image-level"):
        pre_imgs = pre_imgs.to(device)
        post_imgs = post_imgs.to(device)

        logits = model(pre_imgs, post_imgs)
        probs = torch.sigmoid(logits)
        mean_prob = probs.mean(dim=[2,3])          # average over spatial dims
        pred_label = (mean_prob > 0.5).float()

        image_true.append(class_labels.numpy())
        image_pred.append(pred_label.cpu().numpy())
        image_prob.append(mean_prob.cpu().numpy())

image_true = np.concatenate(image_true).ravel()
image_pred = np.concatenate(image_pred).ravel()
image_prob = np.concatenate(image_prob).ravel()

img_acc = accuracy_score(image_true, image_pred)
img_f1 = f1_score(image_true, image_pred, pos_label=1)

print("\nImage-level (binary: damaged building or not)")
print(f"Accuracy : {img_acc*100:.2f}%")
print(f"F1-score : {img_f1:.4f}")

# ------------------------------------------------------------
# 8. Save results to file (useful for Kaggle submission/records)
# ------------------------------------------------------------
results = {
    "pixel_accuracy": float(acc),
    "pixel_auc": float(auc) if not np.isnan(auc) else None,
    "pixel_mae": float(mae),
    "pixel_f1_macro": float(f1_macro),
    "pixel_f1_weighted": float(f1_weighted),
    "image_accuracy": float(img_acc),
    "image_f1": float(img_f1),
}

import json
with open("test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nAll metrics saved to test_metrics.json")
print("Done!")